# Classic ML Models
* Target of this notebook is exploring the capabilities of classic ML regressors from SKLearn. 
* They certainly can't keep up with GBDTs but might add some variance in an ensemble.
* We'll apply minimal preprocessing and feature engineering (scaling + one-hot-encoding + polynomials)
* We'll combine training with original Abalone data

# Dependencies & Settings

In [ ]:
import math
from pathlib import Path
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import (model_selection, metrics, linear_model, preprocessing, 
                     pipeline, compose, dummy, ensemble, kernel_ridge, 
                     neighbors, svm, isotonic)
import lightgbm as lgb

In [ ]:
from warnings import simplefilter
simplefilter("ignore", category=FutureWarning)

In [ ]:
%%html
<style>
table {float:left}
</style>

# Files

In [ ]:
df_train = pd.read_csv("/kaggle/input/playground-series-s4e4/train.csv")
print(f'{df_train.shape=}')

df_test = pd.read_csv("/kaggle/input/playground-series-s4e4/test.csv")
print(f'{df_test.shape=}')

df_original = pd.read_csv("/kaggle/input/abalone-dataset/abalone.csv")
print(f'{df_original.shape=}')

In [ ]:
# modify original column names to match train/test
df_original.columns = df_train.drop('id', axis=1).columns

# make life easier for LGBM
df_train['Sex'] = df_train['Sex'].astype("category")  
df_test['Sex'] = df_test['Sex'].astype("category")  
df_original['Sex'] = df_original['Sex'].astype("category")  

# Merge for Training


In [ ]:
_df_merged = pd.concat([
    df_train.drop(['id'], axis=1),
    df_original
]).reset_index(drop=True)

df_features = _df_merged.drop(['Rings'], axis=1)
ser_targets = _df_merged['Rings']
print(f'{df_features.shape=}\n{ser_targets.shape=}')

# Preprocessing

In [ ]:
metric_cols = ['Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight']
categorical_cols = ['Sex']
assert set(df_features.columns) == set(metric_cols+categorical_cols)

In [ ]:
metric_transformer = pipeline.Pipeline([
    ('poly', preprocessing.PolynomialFeatures(degree=2)),
    ('scale', preprocessing.StandardScaler()),
])

preprocessor = compose.ColumnTransformer(
    transformers=[
        ('metric', metric_transformer, metric_cols),
        ('categorical', preprocessing.OneHotEncoder(), categorical_cols)
    ])

n_features_after_preprocessing = preprocessor.fit_transform(df_features).shape[1]
print(f'{n_features_after_preprocessing=}')

preprocessor

# CV Scoring 

In [ ]:
def score_pipeline(pipeline: pipeline.Pipeline) -> float:
    
    arr_oof_predictions = np.zeros(len(ser_targets))
    kf = model_selection.KFold(
        n_splits=5, shuffle=True, random_state=42
    )
    for fold_no, (train_idx, val_idx) in enumerate(kf.split(df_features)):
        print(f"Starting fold {fold_no}")

        df_train_fold = df_features.iloc[train_idx]
        df_val_fold = df_features.iloc[val_idx]

        ser_targets_train_fold = ser_targets.iloc[train_idx]
        ser_targets_val_fold = ser_targets.iloc[val_idx]

        pipeline.fit(df_train_fold, ser_targets_train_fold)

        arr_oof_predictions[val_idx] = pipeline.predict(df_val_fold)

    arr_oof_predictions = arr_oof_predictions.clip(min=0)

    rmsle = metrics.mean_squared_log_error(y_true=ser_targets,
                                           y_pred=arr_oof_predictions,
                                           squared=False)  # True would get us the MSLE
    print(f'{rmsle=}')
    return rmsle

In [ ]:
df_results = pd.DataFrame()

# BayesianRidge

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.BayesianRidge())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'linear_model.BayesianRidge', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# Automatic Relevance Determination Regression (ARD)

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.ARDRegression())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'linear_model.ARDRegression', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# AdaBoostRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', ensemble.AdaBoostRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'ensemble.AdaBoostRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# RandomForestRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', ensemble.RandomForestRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'ensemble.RandomForestRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# HistGradientBoostingRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', ensemble.HistGradientBoostingRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'ensemble.HistGradientBoostingRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# LinearRegression

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.LinearRegression())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'linear_model.LinearRegression', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# SGDRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.SGDRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'linear_model.SGDRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# ElasticNet (Linear with L1 + L2 Regularization)

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.ElasticNet())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'linear_model.ElasticNet', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# KNeighborsRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', neighbors.KNeighborsRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'neighbors.KNeighborsRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# Support Vector Machine Regression

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', svm.SVR())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'svm.SVR', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

to better assess the performance of the models trained above, we also train...
* a DummyRegressor that always predicts mean target
* an LGBM Regressor with default settings

# DummyRegressor (mean)


In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', dummy.DummyRegressor(strategy="mean"))
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'dummy.DummyRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# LGBM Regressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', lgb.LGBMRegressor())
])

display(pipe)

start_time = time.time()
rmsle = score_pipeline(pipe)

ser_results = pd.Series({'estimator': 'LGBMRegressor', 
           'rmsle': rmsle,
           'duration': time.time() - start_time})

df_results = pd.concat([df_results, pd.DataFrame([ser_results])], ignore_index=True)

# Results

In [ ]:
df_results.sort_values('rmsle').style.background_gradient(axis=0, cmap='RdYlGn_r')

* As expected, the classic ML techniques can't keep up with a sota GBDT.
* RandomForestRegressor and HistGradientBoostingRegressor score quite well, but probably won't add much diversity to an ensemble full of GBDTs.
    * Since HistGradientBoostingRegressor trains very, it might be worth optimizing.
* SVR and the linear models (LinearRegression, ARDRegression, BayesianRidge) score quite well.
    * SVR scores best among the classic models but takes a lot of time to train. Since we don't need a GPU (wouldn't help anyway), we may still try optimizing.
    * The linear models are trained very fast, so some Bayesian optimization of hyperparameters might be worth trying as well.